# Homework 01 — Transaction, Concurrency & Recovery (MySQL / InnoDB)
**ชื่อ:** นางสาวปุญญิศา จันทร์เรือง · **รหัสนักศึกษา:** 664245040

แล็บภาคปฏิบัติฐานข้อมูลขั้นสูงด้วย MySQL (InnoDB) และ Python ครอบคลุม
1. Commit / Rollback (Atomicity)
2. Race Condition (Lost Update) และการแก้ด้วย `SELECT ... FOR UPDATE`
3. Deadlock
4. Isolation Level
5. Redo Log (Durability)

> รันบน Google Colab: กด **Runtime → Run all** ตามลำดับจากบนลงล่าง

## 0. ติดตั้งและเริ่ม MySQL Server

In [ ]:
!apt-get -qq update > /dev/null
!DEBIAN_FRONTEND=noninteractive apt-get -qq install mysql-server > /dev/null
!pip -q install mysql-connector-python
!service mysql start
!mysql -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY 'root'; FLUSH PRIVILEGES;"
!mysql -uroot -proot -e "SELECT VERSION();" 2>/dev/null

In [ ]:
import mysql.connector, threading, time

def conn(db="lab", autocommit=False):
    c = mysql.connector.connect(host="127.0.0.1", user="root", password="root", database=db)
    c.autocommit = autocommit
    return c

root = mysql.connector.connect(host="127.0.0.1", user="root", password="root", autocommit=True)
cur = root.cursor()
cur.execute("DROP DATABASE IF EXISTS lab"); cur.execute("CREATE DATABASE lab")
cur.execute("""CREATE TABLE lab.account(
    id INT PRIMARY KEY, owner VARCHAR(50), balance INT) ENGINE=InnoDB""")
cur.execute("INSERT INTO lab.account VALUES (1,'Alice',1000),(2,'Bob',1000)")

def show(title="account"):
    c = conn(autocommit=True); k = c.cursor()
    k.execute("SELECT * FROM account ORDER BY id"); print(title, k.fetchall()); c.close()

def reset():
    k = root.cursor(); k.execute("UPDATE lab.account SET balance=1000"); 
show("เริ่มต้น")

## 1. Commit / Rollback (Atomicity)
โอนเงิน Alice → Bob 300 บาท ต้อง **สำเร็จทั้งคู่หรือไม่สำเร็จเลย**
กรณีที่ 1: commit ปกติ · กรณีที่ 2: เกิดข้อผิดพลาดกลางทาง แล้ว rollback

In [ ]:
reset()
c = conn(); k = c.cursor()
k.execute("UPDATE account SET balance=balance-300 WHERE id=1")
k.execute("UPDATE account SET balance=balance+300 WHERE id=2")
c.commit(); show("หลัง COMMIT  ")

reset()
c = conn(); k = c.cursor()
try:
    k.execute("UPDATE account SET balance=balance-300 WHERE id=1")
    raise RuntimeError("ระบบล่มก่อนเพิ่มเงินให้ Bob")
    k.execute("UPDATE account SET balance=balance+300 WHERE id=2")
    c.commit()
except Exception as e:
    print("ข้อผิดพลาด:", e); c.rollback()
show("หลัง ROLLBACK")

**สรุป:** หลัง ROLLBACK ยอดเงินกลับเป็นค่าเดิม (1000, 1000) แสดงว่าธุรกรรมมีคุณสมบัติ Atomicity

## 2. Race Condition (Lost Update)
หลาย thread อ่านยอด → คำนวณใน Python → เขียนกลับ โดยไม่ล็อกแถว ทำให้การอัปเดตบางครั้งหายไป
ทดลองให้ 10 thread เพิ่มเงินให้ Alice คนละ 100 บาท (ควรได้ 1000 + 1000 = **2000**)

In [ ]:
def deposit_unsafe():
    c = conn(); k = c.cursor()
    k.execute("SELECT balance FROM account WHERE id=1")
    bal = k.fetchone()[0]
    time.sleep(0.05)                      # จำลองเวลาประมวลผล
    k.execute("UPDATE account SET balance=%s WHERE id=1", (bal+100,))
    c.commit(); c.close()

def deposit_safe():
    c = conn(); k = c.cursor()
    k.execute("SELECT balance FROM account WHERE id=1 FOR UPDATE")   # ล็อกแถว
    bal = k.fetchone()[0]
    time.sleep(0.05)
    k.execute("UPDATE account SET balance=%s WHERE id=1", (bal+100,))
    c.commit(); c.close()

def run(fn, n=10):
    ts = [threading.Thread(target=fn) for _ in range(n)]
    [t.start() for t in ts]; [t.join() for t in ts]

reset(); run(deposit_unsafe); show("ไม่ล็อก (คาด 2000) ")
reset(); run(deposit_safe);   show("FOR UPDATE (2000)   ")

**สรุป:** แบบไม่ล็อกยอดจะน้อยกว่า 2000 (Lost Update) ส่วน `SELECT ... FOR UPDATE` ทำให้ธุรกรรมต่อคิวกัน ได้ 2000 ถูกต้อง

## 3. Deadlock
Transaction A ล็อกแถว 1 แล้วขอแถว 2 ในขณะที่ B ล็อกแถว 2 แล้วขอแถว 1 → รอกันเอง
InnoDB ตรวจพบและ **rollback ฝั่งใดฝั่งหนึ่ง** (error 1213)

In [ ]:
barrier = threading.Barrier(2)
result = {}

def tx(name, first, second):
    c = conn(); k = c.cursor()
    try:
        k.execute("UPDATE account SET balance=balance+1 WHERE id=%s", (first,))
        barrier.wait()                    # ให้ทั้งสองได้ล็อกแรกก่อน
        time.sleep(0.2)
        k.execute("UPDATE account SET balance=balance+1 WHERE id=%s", (second,))
        c.commit(); result[name] = "COMMIT"
    except mysql.connector.Error as e:
        c.rollback(); result[name] = f"ROLLBACK (errno {e.errno}: {e.msg[:40]}...)"
    finally:
        c.close()

reset()
a = threading.Thread(target=tx, args=("A",1,2)); b = threading.Thread(target=tx, args=("B",2,1))
a.start(); b.start(); a.join(); b.join()
print(result)

In [ ]:
k = root.cursor(); k.execute("SHOW ENGINE INNODB STATUS"); s = k.fetchall()[0][2]
i = s.find("LATEST DETECTED DEADLOCK"); print(s[i:i+1500] if i>=0 else "ไม่พบ")

**วิธีป้องกัน:** ล็อกแถวตามลำดับเดียวกันทุก transaction (เช่น id น้อยไปมาก) และทำ transaction ให้สั้น

## 4. Isolation Level
ทดสอบ **Non-repeatable read**: Transaction 1 อ่านค่าสองครั้ง ระหว่างนั้น Transaction 2 แก้ค่าและ commit
- `READ COMMITTED` → อ่านรอบที่สองเห็นค่าใหม่
- `REPEATABLE READ` (ค่าเริ่มต้นของ InnoDB) → เห็นค่าเดิมตลอด transaction

In [ ]:
def test(level):
    reset()
    t1 = conn(); k1 = t1.cursor()
    k1.execute(f"SET SESSION TRANSACTION ISOLATION LEVEL {level}")
    k1.execute("START TRANSACTION")
    k1.execute("SELECT balance FROM account WHERE id=1"); r1 = k1.fetchone()[0]
    t2 = conn(); k2 = t2.cursor()
    k2.execute("UPDATE account SET balance=5000 WHERE id=1"); t2.commit(); t2.close()
    k1.execute("SELECT balance FROM account WHERE id=1"); r2 = k1.fetchone()[0]
    t1.rollback(); t1.close()
    print(f"{level:17} อ่านครั้งที่ 1 = {r1}, ครั้งที่ 2 = {r2}",
          "-> Non-repeatable read" if r1 != r2 else "-> อ่านซ้ำได้ค่าเดิม")

for lv in ["READ UNCOMMITTED","READ COMMITTED","REPEATABLE READ","SERIALIZABLE"]:
    test(lv)

| Isolation Level | Dirty Read | Non-repeatable Read | Phantom Read |
|---|---|---|---|
| READ UNCOMMITTED | เกิดได้ | เกิดได้ | เกิดได้ |
| READ COMMITTED | ไม่เกิด | เกิดได้ | เกิดได้ |
| REPEATABLE READ (InnoDB) | ไม่เกิด | ไม่เกิด | ไม่เกิด (ใช้ MVCC + next-key lock) |
| SERIALIZABLE | ไม่เกิด | ไม่เกิด | ไม่เกิด |

## 5. Redo Log (Durability)
InnoDB เขียนการเปลี่ยนแปลงลง **redo log** ก่อน commit (Write-Ahead Logging) หากเซิร์ฟเวอร์ล่ม จะ replay log เพื่อกู้ข้อมูลที่ commit แล้ว
ขั้นแรกดูค่าที่เกี่ยวข้อง

In [ ]:
k = root.cursor()
for v in ["innodb_flush_log_at_trx_commit","innodb_redo_log_capacity","innodb_log_file_size"]:
    k.execute("SHOW VARIABLES LIKE %s", (v,)); print(k.fetchall())
k.execute("SHOW ENGINE INNODB STATUS"); s = k.fetchall()[0][2]
i = s.find("LOG"); print(s[i:i+600])

ทดลองจำลองการล่ม: commit แล้วฆ่า process `mysqld` แบบ `kill -9` จากนั้นเริ่มใหม่ ข้อมูลที่ commit แล้วต้องยังอยู่

In [ ]:
reset()
c = conn(); k = c.cursor()
k.execute("UPDATE account SET balance=7777 WHERE id=1"); c.commit()     # commit แล้ว
c2 = conn(); k2 = c2.cursor()
k2.execute("UPDATE account SET balance=1 WHERE id=2")                    # ยังไม่ commit
print("ก่อนล่ม: Alice=7777 (commit แล้ว), Bob=1 (ยังไม่ commit)")

In [ ]:
!pkill -9 mysqld
!sleep 3
!service mysql start
!sleep 5

In [ ]:
root = mysql.connector.connect(host="127.0.0.1", user="root", password="root", autocommit=True)
show("หลังกู้คืน")

**สรุป:** หลังเกิดการล่มและกู้คืน Alice = 7777 (commit แล้ว → ยังอยู่ด้วย redo log) ส่วน Bob กลับเป็น 1000 (ยังไม่ commit → ถูก rollback)

## สรุปภาพรวม
| หัวข้อ | กลไกของ InnoDB | ผลที่ได้ |
|---|---|---|
| Atomicity | undo log / rollback | ทำครบหรือไม่ทำเลย |
| Race Condition | row lock (`FOR UPDATE`) | ป้องกัน Lost Update |
| Deadlock | wait-for graph detection | rollback ฝั่งเหยื่อ ฝั่งอื่นผ่านได้ |
| Isolation | MVCC + lock | เลือกระดับความเข้มงวดได้ |
| Durability | redo log (WAL) | กู้ข้อมูลที่ commit หลังล่ม |